# Analyzing UMBL2022FEB cell data

2024/07/19

Unpacking the UMBL2022FEB dataset, starting with the processed `.pkl` files summarized courtesy of Iaroslav Kovalchuk

Andrew Weng

In [8]:
import os, sys

# Move the path up a level to be able to index into source files
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../')
    sys.path.insert(0, 'src/')

import numpy as np
import pickle
import pandas as pd
from matplotlib import pyplot as plt
from src import plotter as plotter

%load_ext autoreload

plotter.initialize(plt)

target_dir = os.getcwd()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
def read_pickle(path):
    """ Easily load data from Pickle files"""
    with open(path, 'rb') as f:
        data = pickle.load(f)

    return data

In [71]:
def fetch_dataframe(data_dict, filetype):
    """
    Fetch DataFrame from a dictionary of dataframes based on a type

    Args:
    data_dict (dict): Dictionary of DataFrames
       this is the standard format for the data files given by Iaro
    filetype (str): Type of DataFrame to fetch
       options are 'formation_cycle', 'formation_tap', 'formation_aging', 'cycling', 'aux'

    Returns:
    DataFrame: DataFrame of the specified type

    """

    assert filetype in ['formation_cycle', 'formation_tap', 'formation_aging', 'cycling', 'aux'], \
        f"Invalid filetype: {filetype}"

    key_list = list(data_dict.keys())

    result_list = []

    if filetype == 'formation_cycle':
        for key in key_list:
            if any([x in key for x in ['FORMBASE', 'FORMFAST']]):
                result_list.append(key)
    if filetype == 'formation_tap':
        for key in key_list:
            if 'FORMTAP' in key:
                result_list.append(key)
    if filetype == 'formation_aging':
        for key in key_list:
            if 'FORMAGING' in key:
                result_list.append(key)
    if filetype == 'cycling':
        for key in key_list:
            if 'CYC' in key:
                result_list.append(key)
    if filetype == 'aux':
        for key in key_list:
            if 'AuxDat' in key:
                result_list.append(key)

    assert len(result_list) <= 1, f"Multiple keys found: {result_list}"
    assert len(result_list) >= 1, f"No keys found"

    key = result_list[0]

    return data_dict[key]

# Set the data directory path

In [10]:
ROOT_PATH = '/Users/aweng/Documents/PROJ_UMBL2022FEB/pouch/'

# Inspect the metadata file

In [11]:
df_metadata = pd.read_csv('data/UMBL2022FEB_Metadata.csv',delimiter=' *, *')
df_metadata

/var/folders/qr/bx1pzh1x6nnbdrjvw2qhqkf40000gn/T/ipykernel_40500/3267469806.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_metadata = pd.read_csv('data/UMBL2022FEB_Metadata.csv',delimiter=' *, *')


,Unnamed: 0,device_id,formation_temperature_c,formation_pressure_psi,graphite_type,formation_type,cyc_temperature_c,cyc_condition
0,0,151802,RT,5P0PSI,1518,FORMBASE_1,45C,1C/1C
1,1,151803,RT,5P0PSI,1518,FORMBASE_1,45C,1C/1C
2,2,151804,RT,5P0PSI,1518,FORMBASE_1,45C,1C/1C
3,3,151805,RT,5P0PSI,1518,FORMBASE_1,45C,1C/1C
4,4,151806,RT,5P0PSI,1518,FORMBASE_1,45C,1C/1C
5,5,152001,RT,5P0PSI,1520,FORMBASE_1,45C,1C/1C
6,6,152002,RT,5P0PSI,1520,FORMBASE_1,45C,1C/1C
7,7,152004,RT,5P0PSI,1520,FORMBASE_1,45C,1C/1C
8,8,152005,RT,5P0PSI,1520,FORMFAST_2,45C,1C/1C
9,9,152006,RT,5P0PSI,1520,FORMFAST_2,45C,1C/1C


# Assemble Device ID lists corresponding to each study

In [12]:
# Assemble Device ID lists
device_id_control = df_metadata[(df_metadata['formation_temperature_c'] == "RT") &
                                (df_metadata['formation_pressure_psi']  == "5P0PSI") &
                                (df_metadata['graphite_type']           == 1520) &
                                (df_metadata['formation_type']          == "FORMBASE_1")] \
                                ['device_id']

device_id_temp =    df_metadata[(df_metadata['formation_temperature_c'] == "55C") &
                                (df_metadata['formation_pressure_psi']  == "5P0PSI") &
                                (df_metadata['graphite_type']           == 1520) &
                                (df_metadata['formation_type']          == "FORMBASE_1")] \
                                ['device_id']

device_id_pressure = df_metadata[(df_metadata['formation_temperature_c'] == "RT") &
                                (df_metadata['formation_pressure_psi']   == "0P0PSI") &
                                (df_metadata['graphite_type']            == 1520) &
                                (df_metadata['formation_type']           == "FORMBASE_1")] \
                                ['device_id']

device_id_fast1 = df_metadata[(df_metadata['formation_temperature_c']   == "RT") &
                                (df_metadata['formation_pressure_psi']  == "5P0PSI") &
                                (df_metadata['graphite_type']           == 1520) &
                                (df_metadata['formation_type']          == "FORMFAST_1")] \
                                ['device_id']

device_id_fast2 = df_metadata[(df_metadata['formation_temperature_c']   == "RT") &
                                (df_metadata['formation_pressure_psi']  == "5P0PSI") &
                                (df_metadata['graphite_type']           == 1520) &
                                (df_metadata['formation_type']          == "FORMFAST_2")] \
                                ['device_id']

device_id_control = df_metadata[(df_metadata['formation_temperature_c'] == "RT") &
                                (df_metadata['formation_pressure_psi']  == "5P0PSI") &
                                (df_metadata['graphite_type']           == 1518) &
                                (df_metadata['formation_type']          == "FORMBASE_1")] \
                                ['device_id']

device_id_control

0    151802
1    151803
2    151804
3    151805
4    151806
Name: device_id, dtype: int64

# Analyze a single cell

In [26]:
target_device_id = 152017

full_path_raw = ROOT_PATH + f'UMBL2022FEB_CELL{target_device_id}.pkl'
full_path_processed = ROOT_PATH + f'UMBl2022FEB_CELL{target_device_id}_processed.pkl'

In [37]:
data_raw = read_pickle(full_path_raw)
data_processed = read_pickle(full_path_processed)

In [72]:
df = fetch_dataframe(data_raw, 'cycling')

In [73]:
df

,[Arbin] AC Impedance (Ω),[Arbin] AC Impedance Phase Angle (°),[Arbin] Charge Capacity (Ah),[Arbin] Charge Energy (Wh),[Arbin] Current (A),[Arbin] Cycle Number,[Arbin] Datapoint Number,[Arbin] Date Time (excel format),[Arbin] Discharge Capacity (Ah),[Arbin] Discharge Energy (Wh),...,i_cycle_num,Datapoint Number,h_datapoint_time,h_discharge_capacity,h_discharge_energy,h_potential,h_step_index,h_step_time,h_test_time,h_datapoint_datetime
0,0.0,0.0,0.000000,0.000000,0.0,1,1,44777.466956,0.000000,0.0000,...,1.0,1.0,2022-08-04 15:12:25+00:00,0.000000,0.0000,3.346937,1.0,0.063622,6.362190e-02,2022-08-04 15:12:25+00:00
1,0.0,0.0,0.000000,0.000000,0.0,1,2,44777.467072,0.000000,0.0000,...,1.0,2.0,2022-08-04 15:12:35+00:00,0.000000,0.0000,3.347099,1.0,10.076638,1.007664e+01,2022-08-04 15:12:35+00:00
2,0.0,0.0,0.000000,0.000000,0.0,1,3,44777.467187,0.000000,0.0000,...,1.0,3.0,2022-08-04 15:12:45+00:00,0.000000,0.0000,3.346937,1.0,20.088669,2.008867e+01,2022-08-04 15:12:45+00:00
3,0.0,0.0,0.000000,0.000000,0.0,1,4,44777.467303,0.000000,0.0000,...,1.0,4.0,2022-08-04 15:12:55+00:00,0.000000,0.0000,3.346937,1.0,30.100033,3.010003e+01,2022-08-04 15:12:55+00:00
4,0.0,0.0,0.000000,0.000000,0.0,1,5,44777.467419,0.000000,0.0000,...,1.0,5.0,2022-08-04 15:13:05+00:00,0.000000,0.0000,3.346937,1.0,40.111594,4.011159e+01,2022-08-04 15:13:05+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019234,0.0,0.0,1.203189,5.008445,0.0,2074,2019255,44994.633785,1.605553,5.4644,...,2074.0,2019255.0,2023-03-09 20:12:39+00:00,1.605553,5.4644,3.233272,31.0,10606.829453,1.802681e+07,2023-03-09 20:12:39+00:00
2019235,0.0,0.0,1.203189,5.008445,0.0,2074,2019256,44994.633900,1.605553,5.4644,...,2074.0,2019256.0,2023-03-09 20:12:49+00:00,1.605553,5.4644,3.233272,31.0,10616.838673,1.802682e+07,2023-03-09 20:12:49+00:00
2019236,0.0,0.0,1.203189,5.008445,0.0,2074,2019257,44994.634016,1.605553,5.4644,...,2074.0,2019257.0,2023-03-09 20:12:59+00:00,1.605553,5.4644,3.233434,31.0,10626.860160,1.802683e+07,2023-03-09 20:12:59+00:00
2019237,0.0,0.0,1.203189,5.008445,0.0,2074,2019258,44994.634132,1.605553,5.4644,...,2074.0,2019258.0,2023-03-09 20:13:09+00:00,1.605553,5.4644,3.233434,31.0,10636.877721,1.802684e+07,2023-03-09 20:13:09+00:00


In [80]:
df.columns

Index(['[Arbin] AC Impedance (Ω)', '[Arbin] AC Impedance Phase Angle (°)',
       '[Arbin] Charge Capacity (Ah)', '[Arbin] Charge Energy (Wh)',
       '[Arbin] Current (A)', '[Arbin] Cycle Number',
       '[Arbin] Datapoint Number', '[Arbin] Date Time (excel format)',
       '[Arbin] Discharge Capacity (Ah)', '[Arbin] Discharge Energy (Wh)',
       '[Arbin] dV/dT (V/s)', '[Arbin] Internal Resistance (Ω)',
       '[Arbin] Is Fast Capture', '[Arbin] Potential (V)',
       '[Arbin] Step Index', '[Arbin] Step Time (s)', '[Arbin] Test Time (s)',
       '[Arbin] Test_ID 0', 'Test Cumulative Capacity (Ah)',
       'Test Cumulative Energy (Wh)', 'Test Net Capacity (Ah)',
       'Test Net Energy (Wh)', 'dQ/dV (Ah/V)', 'dV/dt (V/s)', 'Power (W)',
       'Current Cycle Net Capacity (Ah)', 'Current Cycle Net Energy (Wh)',
       'h_charge_capacity', 'h_charge_energy', 'h_current', 'i_cycle_num',
       'Datapoint Number', 'h_datapoint_time', 'h_discharge_capacity',
       'h_discharge_energy', 'h_

In [85]:
df_agg = df.groupby('i_cycle_num').agg({'h_test_time': 'max',
                               'Test Cumulative Capacity (Ah)': 'max',
                               'h_discharge_capacity': 'max'})